In [1]:
!pip install scikit-learn-extra
# import the libraries
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pandas.tseries.offsets import DateOffset

# Rest of your imports and code should follow

from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import silhouette_score, pairwise_distances, davies_bouldin_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
import seaborn as sns
from mpl_toolkits.mplot3d import Axes3D
from google.colab import files
from google.colab import drive
from sklearn.svm import SVC
from sklearn.tree import plot_tree
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sentence_transformers import SentenceTransformer
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

from sklearn.model_selection import StratifiedKFold, cross_val_predict


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 15.4 MB/s eta 0:00:00


## Test

In [3]:
from google.colab import drive
drive.mount('/content/drive')

df_test_path = '/content/drive/MyDrive/למידת מכונה/test.csv'
df = pd.read_csv(df_test_path)

df.head()

Mounted at /content/drive


Data Types

In [4]:
df.replace('?', np.nan, inplace=True) #replace all the '?' to nan
# Convert to datetime
df['post_datetime'] = pd.to_datetime(df['post_datetime'], errors='coerce')
df['birthday'] = pd.to_datetime(df['birthday'], errors='coerce')
df['account_creation_date'] = pd.to_datetime(df['account_creation_date'], errors='coerce')

# Convert to numeric
numeric_cols = [
    'post_likes', 'posts_retweets', 'followers',
    'previous_posts_count'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Convert sentiment and type to category
df['type'] = df['type'].astype('category')
df['checkmark'] = df['checkmark'].astype('category')
df['timezone'] = df['timezone'].astype('category')

#convert text fields like username/text to string
df['username'] = df['username'].astype('string')
df['text'] = df['text'].astype('string')

df.dtypes

,0
id,object
text,string[python]
type,category
post_datetime,datetime64[ns]
timezone,category
post_likes,int64
posts_retweets,int64
embedded_content_url,object
username,string[python]
checkmark,category


Duplicate Records

In [5]:
# Sort by user, text, and datetime
df_sorted = df.sort_values(by=['username', 'text', 'post_datetime'])

# Create a shifted datetime per user+text group
df_sorted['prev_post_time'] = df_sorted.groupby(['username', 'text'])['post_datetime'].shift()

# Calculate time difference in seconds
df_sorted['time_diff'] = (df_sorted['post_datetime'] - df_sorted['prev_post_time']).dt.total_seconds()

# Filter where the time diff is less than or equal to 600 seconds
duplicates = df_sorted[df_sorted['time_diff'].between(0, 600)]
print(f"Found {len(duplicates)} duplicate samples by  user, text, and datetime.")

# Find duplicates based on text + datetime
dup_mask = df.duplicated(subset=['text', 'post_datetime'], keep=False)
duplicates_text_time = df[dup_mask]
print(f"Found {len(duplicates_text_time)} posts with the same text and datetime.")

#Sort by Username and text
user_text_type_counts = df.groupby(['username', 'text'])['type'].nunique()
reused_by_user = user_text_type_counts[user_text_type_counts > 1]

print(f"Found {len(reused_by_user)} user-text pairs reused in multiple types.\n")





Found 0 duplicate samples by  user, text, and datetime.
Found 0 posts with the same text and datetime.
Found 0 user-text pairs reused in multiple types.



Missing Data

In [6]:
from datetime import timedelta
#Timezone
df['timezone'] = df['timezone'].fillna('UTC') #missing timezone> UTC
mean_hour_utc = int(df['post_datetime'].dropna().dt.hour.mean()) #calculate the mean time
#Timezone map
tz_offset = {
    'UTC':  0, 'GMT':  0,
    'EST': -5, 'PST': -8,
    'CET': +1, 'EET': +2, 'MSK': +3,
    'AEST': +10, 'KST': +9, 'JST': +9
}

#calculate by local timezone
def generate_datetime_utc(tz):
    offset = tz_offset.get(tz, 0)
    random_days = np.random.randint(0, 365)
    base_date = pd.Timestamp.today().normalize() - pd.Timedelta(days=random_days)
    local_dt = base_date + pd.Timedelta(hours=mean_hour_utc)
    utc_dt = local_dt - pd.Timedelta(hours=offset)
    return utc_dt
missing_mask = df['post_datetime'].isna()
df.loc[missing_mask, 'post_datetime'] = df.loc[missing_mask, 'timezone'].apply(generate_datetime_utc)

#update the local time
offset_hours = df['timezone'].map(tz_offset).fillna(0)
df['local_dt'] = df['post_datetime'] + pd.to_timedelta(offset_hours, unit='h')
df['local_hour'] = df['local_dt'].dt.hour


#age
today = pd.Timestamp.today()
df['age'] = ((today - df['birthday']).dt.days // 365)
mean_age = int(df['age'].dropna().mean())#Calculate average age (in years) from existing birthdays

#Generate random birthdays based on the average age
def generate_random_birthday(avg_age):
    # Add small randomness in days (up to 1 year)
    random_days = np.random.randint(0, 365)
    return today - pd.Timedelta(days=(avg_age * 365 + random_days))

#Fill missing birthday values with random birthday
missing_birthday_mask = df['birthday'].isna()
df.loc[missing_birthday_mask, 'birthday'] = [
    generate_random_birthday(mean_age) for _ in range(missing_birthday_mask.sum())
]

#Recalculate age for all rows (including newly filled)
df['age'] = ((today - df['birthday']).dt.days // 365)


#numeric variables
zero_fill_cols = ['post_likes', 'posts_retweets', 'followers', 'previous_posts_count']
df[zero_fill_cols] = df[zero_fill_cols].fillna(0)

#checkmark
df['checkmark'] = df['checkmark'].fillna('none')

#profile_picture
df['has_profile_pic'] = df['profile_picture'].notna() & df['profile_picture'].str.strip().ne('')
df['has_profile_pic'] = df['has_profile_pic'].astype(int)
#email
df['email'] = df['email'].fillna('').astype(str)
#username
df['username'] = df['username'].fillna('').astype(str)

#account_creation_date, years on platform
today = pd.Timestamp.today()
df['days_on_platform'] = (today - df['account_creation_date']).dt.days
valid_rate_mask = (df['previous_posts_count'].notna()) & (df['days_on_platform'] > 0)

# קצב פוסטים ליום למשתמשים תקינים
df['posting_rate'] = np.nan
df.loc[valid_rate_mask, 'posting_rate'] = df.loc[valid_rate_mask, 'previous_posts_count'] / df.loc[valid_rate_mask, 'days_on_platform']
avg_posts_per_day = df['posting_rate'].mean()

# רשומות עם תאריך פתיחה חסר
missing_mask = df['account_creation_date'].isna()
estimated_days = (df.loc[missing_mask, 'previous_posts_count'] / avg_posts_per_day).clip(lower=30, upper=5*365) #calculate number of days back
df.loc[missing_mask, 'account_creation_date'] = today - pd.to_timedelta(estimated_days, unit='D') #create the account creatain date

#update years_on_platform
df['years_on_platform'] = ((today - df['account_creation_date']).dt.days // 365)


In [7]:
df.isna().mean() * 100 #find all the empty fields

,0
id,0.00
text,0.00
type,0.00
post_datetime,0.00
timezone,0.00
post_likes,0.00
posts_retweets,0.00
embedded_content_url,77.80
username,0.00
checkmark,0.00


Catagories

In [8]:
#ages
df['birthday'] = pd.to_datetime(df['birthday'], errors='coerce')
df['age'] = (pd.Timestamp.today() - df['birthday']).dt.days // 365

bins   = [0, 18, 25, 35, 45, 55, 65, 120]
labels = ['<18', '18–24', '25–34', '35–44', '45–54', '55–64', '65+']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)
#years
df['post_datetime'] = pd.to_datetime(df['post_datetime']) #convert to date_time type
df['year'] = df['post_datetime'].dt.year

#posting time of the day
df['hour'] = df['post_datetime'].dt.hour

def get_day_period(hour):
    if 3 <= hour < 7:
        return 'Early Morning' #(03–07)
    elif 7 <= hour < 11:
        return 'Morning' #(07–11)
    elif 11 <= hour < 15:
        return 'Midday' # (11–15)
    elif 15 <= hour < 19:
        return 'Afternoon' # (15–19)
    elif 19 <= hour < 23:
        return 'Evening' # (19–23)
    else:
        return 'Night' # (23–03)

df['day_period'] = df['hour'].apply(get_day_period)


Segmentation

In [9]:

# Has Embedded Content
df['has_embedded_content'] = df['embedded_content_url'].notna().astype(int)

#Has Profile Picture ===
df['has_profile_picture'] = df['profile_picture'].notna().astype(int)

#Profile Picture Type (basic)
df['profile_pic_type'] = df['profile_picture'].apply(
    lambda x: 'default' if pd.notna(x) and 'default' in x else (
              'custom' if pd.notna(x) else 'none'))



Feature Representation

Catgorial Features

In [10]:

# One-hot encode the age_group column to boolean columns
df_age_bools = pd.get_dummies(df['age_group'], prefix='age_group', dtype=int)


# One-hot encode 'day_period'
df_day_bools = pd.get_dummies(df['day_period'], prefix='day_period', dtype=int)

# One-hot encode 'checkmark'
df_checkmark_bools = pd.get_dummies(df['checkmark'], prefix='checkmark', dtype=int)

# Combine everything
df = pd.concat([df, df_age_bools, df_day_bools, df_checkmark_bools], axis=1)

print(df.head(2))

                                     id  \
0  4c059c75-1217-499c-b09f-fd413eb5ae02   
1  9bc6c736-fe28-4d77-ad24-bc1f7f6e223e   

                                                text  type  \
0  i am feeling distracted and not managing to be...  post   
1  im still feeling regretful about not having po...  post   

        post_datetime timezone  post_likes  posts_retweets  \
0 2022-12-17 15:11:00     AEST          56              29   
1 2020-04-27 14:01:00      EST         130              36   

                                embedded_content_url   username checkmark  \
0  https://assets.storage.net/070a755a-b439-4da9-...  @hjrm7nma      none   
1                                                NaN  @user1854      none   

   ... day_period_Afternoon day_period_Early Morning day_period_Evening  \
0  ...                    1                        0                  0   
1  ...                    0                        0                  0   

   day_period_Midday day_period_Mornin

Numeric Features

In [11]:
from sklearn.preprocessing import MinMaxScaler

# Extract hashtags
df['hashtags'] = df['text'].str.findall(r'#\w+').apply(lambda x: x if isinstance(x, list) else [])

# Count number of hashtags
df['hashtag_count'] = df['hashtags'].apply(len)

# Numeric columns
numeric_cols = ['followers', 'posts_retweets', 'previous_posts_count', 'years_on_platform', 'hashtag_count']

# Normalize
scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df[numeric_cols]), columns=numeric_cols)

# 🔐 Reset indices to avoid misalignment
df.reset_index(drop=True, inplace=True)
df_scaled.reset_index(drop=True, inplace=True)

# Merge safely
df = pd.concat([df, df_scaled.add_suffix('_scaled')], axis=1)

# Confirm text is still full
print(df[['text'] + [col + '_scaled' for col in numeric_cols]].head())


                                                text  followers_scaled  \
0  i am feeling distracted and not managing to be...          0.001874   
1  im still feeling regretful about not having po...          0.002678   
2  i feeling very generous i uploading a paid cou...          0.001000   
3  i suddenly feel like i might as well be clutch...          0.000677   
4  i feel that a year from now president ronald r...          0.002640   

   posts_retweets_scaled  previous_posts_count_scaled  \
0               0.001680                     0.025710   
1               0.002085                     0.069012   
2               0.001506                     0.090663   
3               0.000579                     0.097429   
4               0.002607                     0.082544   

   years_on_platform_scaled  hashtag_count_scaled  
0                      0.25                   0.0  
1                      0.55                   0.0  
2                      0.25                   0.0  
3   

Text - TF-IDF

In [12]:
pos = [('happy', 133.92676024338166), ('blessed', 128.7235755582426), ('passionate', 111.74427429428252), ('accepted', 110.65219485906029), ('sweet', 108.52689365092024), ('better', 97.53213526665262), ('loved', 96.29606437950342), ('sure', 95.4940659632029), ('generous', 90.39510074335298), ('thankful', 89.25661786126342), ('important', 88.5425007483484), ('supporting', 87.22002816104197), ('excited', 83.7233319261108), ('positive', 83.02572163426397), ('href', 79.14813242892964), ('confident', 78.4476930213457), ('wonderful', 76.79923235746848), ('comfortable', 76.73062658137762), ('productive', 75.01443231325153), ('proud', 74.40104231115025), ('special', 74.01058496048724), ('amazing', 73.61298028765168), ('lucky', 72.72073246491239), ('relieved', 72.49351409258524), ('joyful', 72.24358622072332), ('family', 72.19451605672758), ('free', 71.23927283608705), ('new', 70.93118605696682), ('safe', 69.97613203102823), ('calm', 68.26915489978089), ('peaceful', 68.1386083796791), ('beloved', 67.9284383381921), ('liked', 67.79640980861151), ('long', 67.69087362615757), ('relaxed', 67.3825714700044), ('successful', 67.32638283938554), ('able', 67.300248264038), ('energetic', 66.82344759137469), ('come', 66.51608298116942), ('hot', 65.85775972778924), ('strong', 65.81356393449488), ('smug', 65.08891718554305), ('sympathetic', 64.91613214164227), ('worthwhile', 63.96992079338226), ('caring', 63.66244475756555), ('honored', 63.503015984243426), ('smart', 62.8301826642537), ('hopeful', 62.59923292164228), ('loving', 62.55704590404821), ('determined', 62.48243848057056)]
neg = [('angry', 127.2091501520244), ('stressed', 126.5560914206212), ('helpless', 125.66379993550773), ('sad', 123.20898531102947), ('bad', 111.76005218734853), ('hated', 111.36352022683225), ('hurt', 111.192881545421), ('lost', 106.19378210601896), ('left', 103.91709489031666), ('kind', 103.36915854430696), ('agitated', 102.48550431154924), ('hate', 102.42609726530077), ('scared', 94.60569797158837), ('started', 92.65770073552109), ('getting', 92.32428018816286), ('felt', 91.56821897573602), ('anxious', 91.33850539300576), ('frustrated', 91.14860259439017), ('having', 90.23947143848551), ('guilty', 87.15677403621942), ('remember', 86.47666439328127), ('depressed', 85.96286283548939), ('tortured', 85.72299230504257), ('lately', 84.7055919076726), ('overwhelmed', 83.3793706650637), ('lonely', 83.14005932046089), ('thing', 82.62997035420946), ('away', 81.04810803349432), ('times', 80.86069644163987), ('said', 79.72937132367379), ('uncomfortable', 79.29909799079361), ('cold', 79.22608045229896), ('afraid', 77.1941206411314), ('try', 76.9312327322695), ('tired', 76.75222261981732), ('boring', 76.5176397340514), ('start', 76.43059093498303), ('stupid', 76.06117295122864), ('exhausted', 75.57619066073494), ('unhappy', 75.35293378222619), ('trying', 75.28464565185249), ('useless', 75.09972991588485), ('insecure', 74.87985135226597), ('annoyed', 74.65766238608703), ('mad', 74.60929565845136), ('shitty', 74.37793370207686), ('sorry', 74.37066014301827), ('completely', 74.30405248328427), ('pain', 73.25876821127987), ('awkward', 72.33216774618083)]
import re
import joblib
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load saved vectorizer and scaler
vectorizer = joblib.load('/content/drive/MyDrive/למידת מכונה/tfidf_vectorizer.pkl')
scaler = joblib.load('/content/drive/MyDrive/למידת מכונה/scaler.pkl')

# Define pos/neg word lists (or load them)
pos_words = [w for w, _ in pos]
neg_words = [w for w, _ in neg]

# Clean function
def clean(text):
    text = str(text).lower()
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'[^a-z\s]', '', text)
    return text.strip()

# Clean test data
df['clean_text'] = df['text'].apply(clean)

# TF-IDF transform using loaded vectorizer (do NOT fit again)
tfidf_new = vectorizer.transform(df['clean_text'])

# Get indices of pos/neg words
feature_names = vectorizer.get_feature_names_out()
pos_idx = [i for i, term in enumerate(feature_names) if term in pos_words]
neg_idx = [i for i, term in enumerate(feature_names) if term in neg_words]

# Compute scores
pos_scores = tfidf_new[:, pos_idx].sum(axis=1).A.ravel()
neg_scores = tfidf_new[:, neg_idx].sum(axis=1).A.ravel()

# Use the same scaler as before
scaled = scaler.transform(pd.DataFrame({'pos_score': pos_scores, 'neg_score': neg_scores}))

# Add scaled scores to test DataFrame
df['pos_score_scaled'] = scaled[:, 0]
df['neg_score_scaled'] = scaled[:, 1]

# Preview
print(df[['text', 'pos_score_scaled', 'neg_score_scaled']].head())


                                                text  pos_score_scaled  \
0  i am feeling distracted and not managing to be...          0.000000   
1  im still feeling regretful about not having po...          0.001519   
2  i feeling very generous i uploading a paid cou...          0.002066   
3  i suddenly feel like i might as well be clutch...          0.000000   
4  i feel that a year from now president ronald r...          0.000000   

   neg_score_scaled  
0          0.000000  
1          0.001804  
2          0.000000  
3          0.000751  
4          0.000000  


/usr/local/lib/python3.11/dist-packages/sklearn/utils/validation.py:2732: UserWarning: X has feature names, but MinMaxScaler was fitted without feature names
  warnings.warn(


Bert model

In [ ]:
texts = df['text'].tolist()
bert_model = SentenceTransformer('all-MiniLM-L6-v2')  # Load same model
X_bert = bert_model.encode(texts, show_progress_bar=True)


df_bert = pd.DataFrame(X_bert, columns=[f'bert_{i}' for i in range(X_bert.shape[1])])


In [ ]:
df_bert = pd.DataFrame(X_bert, columns=[f'bert_{i}' for i in range(X_bert.shape[1])])

selected_features = [
    'followers_scaled', 'posts_retweets_scaled', 'previous_posts_count_scaled', 'hashtag_count_scaled',
    'pos_score_scaled', 'neg_score_scaled',

    # Binary
    'has_profile_pic',

    # Categorical (one-hot encoded)
    'checkmark_blue', 'checkmark_gold', 'checkmark_grey', 'checkmark_none',
    'day_period_Early Morning', 'day_period_Morning', 'day_period_Midday', 'day_period_Afternoon',
    'day_period_Evening', 'day_period_Night',
    'age_group_<18', 'age_group_18–24', 'age_group_25–34', 'age_group_35–44',
    'age_group_45–54', 'age_group_55–64', 'age_group_65+'
]


df_test = pd.concat([df[selected_features], df_bert], axis=1)

In [ ]:
df_test.shape
df_test.head()

Load Trained Model

In [ ]:
from tensorflow.keras.models import load_model

final_model = load_model('/content/drive/MyDrive/למידת מכונה/final_keras_model.h5')

probs = final_model.predict(df_test).ravel()
labels = ['positive' if p >= 0.5 else 'negative' for p in probs]

sentiment_avg = labels_series.value_counts(normalize=True).mul(100).round(2)
print("📊 Sentiment Distribution (predicted by final model):")
print(sentiment_avg)

In [ ]:
df_out = pd.DataFrame({'id': df['id'], 'sentiment': labels})

#df_out.to_csv('7.csv', index=False)
